# 5. Final Model — Ensemble Methods for Gesture Classification

**Course:** TC5035.10 - Capstone project  
**Tecnológico de Monterrey**

**Professor:** Raúl Valente Ramírez Velarde

---
**Team:** 57
> César Miguel Barrientos Robles - A01796615
>
> Alan Julian Rodríguez García - A01796833

---

## Objective

Build and optimize a diverse set of **ensemble models** for the 29-letter LSM hand-sign classification problem, then select the final model that best fits the project's business needs.

Following the activity requirements:

1. Apply both **homogeneous** ensemble strategies (Bagging, Boosting, Gradient Boosting, Random Forest) and **heterogeneous** ensemble strategies (Stacking, Blending).
2. Include **hyperparameter optimization** for the most relevant ensembles.
3. For Stacking / Blending, reuse the **best individual models from the previous phase** (`alternative_models.ipynb`).
4. Synthesize results in a **comparative table** that also includes the individual baselines from Phase I, ordered by the primary metric (F1-Macro), with secondary metrics and training time.
5. Choose the **final model** aligned with the project's business objectives.
6. For the final model, produce significant **diagnostic plots** with interpretation (confusion matrix, ROC, Precision-Recall, feature importance).

Primary metric: **F1-Macro** (treats every letter equally regardless of frequency — every letter matters in fingerspelling).

## 1. Recap — Phase I Individual Models

The previous activity (`alternative_models.ipynb`) compared seven individual classifiers on the same 29-letter LSM problem using mean-pooled MediaPipe keypoints. The validation results were:

| # | Model | F1-Macro | Accuracy | Precision | Recall | Train time (s) |
|---|-------|---------:|---------:|----------:|-------:|---------------:|
| 1 | **MLP** | **0.9425** | 0.9443 | 0.9399 | 0.9471 | 3.38 |
| 2 | **RandomForest** | **0.8971** | 0.8970 | 0.9012 | 0.8959 | 0.39 |
| 3 | LogisticRegression | 0.8781 | 0.9019 | 0.8801 | 0.8808 | 0.29 |
| 4 | SVM (RBF) | 0.8768 | 0.8886 | 0.8898 | 0.8733 | 0.32 |
| 5 | KNN | 0.8407 | 0.8338 | 0.8428 | 0.8418 | 0.003 |
| 6 | DecisionTree | 0.7802 | 0.7913 | 0.7796 | 0.7838 | 0.33 |
| 7 | GaussianNB | 0.6447 | 0.6618 | 0.6538 | 0.6778 | 0.008 |

After randomized hyperparameter search on the top two:

| Model | F1-Macro (baseline) | F1-Macro (tuned) | Best params |
|-------|--------------------:|-----------------:|-------------|
| **MLP (tuned)** | 0.9425 | **0.9415** | `hidden_layer_sizes=(128, 64)`, `activation='tanh'`, `alpha=0.001`, `learning_rate_init=0.001` |
| **RandomForest (tuned)** | 0.8971 | **0.9032** | `n_estimators=200`, `max_depth=50`, `min_samples_split=2`, `min_samples_leaf=1` |

**Key takeaways carried into this notebook:**

- **MLP and Random Forest** are the strongest individual classifiers — they will serve as the base learners for stacking and blending.
- **SVM** is close behind and adds genuine diversity (kernel-based, geometric margin) → also a natural base learner.
- The default MLP already approaches a ceiling — modest tuning gain — so improving further likely requires either richer temporal features or model **combination** (the point of this notebook).
- Tree-based models are the natural homogeneous-ensemble candidates (Bagging, AdaBoost, Gradient Boosting, Random Forest), since they have high variance individually but average out well.

This notebook now layers ensemble strategies on top of those individual learners and re-ranks everything in one combined table.

## 2. Environment and Imports

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.ensemble import (
    AdaBoostClassifier,
    BaggingClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
    StackingClassifier,
)
from sklearn.experimental import enable_halving_search_cv  # noqa: F401
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    auc,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    GridSearchCV,
    HalvingGridSearchCV,
    train_test_split,
)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from manos_hablando.dataset_full import load_full_keypoints
from manos_hablando.dataset_video import normalize_video_sequences

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid")

## 3. Data Loading and Preparation

We keep the **exact same data pipeline** as `alternative_models.ipynb` so the comparison stays apples-to-apples:

1. Load the unified 29-class keypoint dataset (`data/processed/full_keypoints.json`).
2. Apply wrist-centered normalization (`normalize_video_sequences`).
3. **Mean-pool** each variable-length sequence over time → one `(63,)` feature vector per sample.
4. Cap each class at `MAX_PER_CLASS = 500` samples for a fair, fast comparison.
5. Stratified 60 / 20 / 20 train / validation / test split.

In [ ]:
sequences, y_all, encoder = load_full_keypoints()
sequences = normalize_video_sequences(sequences)

# Mean-pool over time → (63,) per sample
X_all = np.stack([seq.mean(axis=0) for seq in sequences]).astype(np.float32)
class_names = list(encoder.classes_)
n_classes = len(class_names)

MAX_PER_CLASS = 500
rng = np.random.default_rng(RANDOM_STATE)
kept_indices: list[int] = []
for cls in np.unique(y_all):
    cls_idx = np.where(y_all == cls)[0]
    if len(cls_idx) > MAX_PER_CLASS:
        cls_idx = rng.choice(cls_idx, size=MAX_PER_CLASS, replace=False)
    kept_indices.extend(cls_idx.tolist())

kept_indices = np.array(sorted(kept_indices))
X = X_all[kept_indices]
y = y_all[kept_indices]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.4,
    random_state=RANDOM_STATE,
    stratify=y,
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=RANDOM_STATE,
    stratify=y_temp,
)

print(f"Classes ({n_classes}): {class_names}")
print(f"Balanced dataset shape: X={X.shape}, y={y.shape}")
print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

### 3.1 Shared evaluation helper

Every model in this notebook is wrapped in a `StandardScaler → estimator` pipeline (some ensembles don't need scaling, but adding it is cheap and keeps the comparison uniform). The helper below fits each pipeline, records training time, and returns a metrics dict measured on both validation and test sets — so we can build the comparative table at the end with one consistent shape.

In [ ]:
def evaluate(name: str, model, fit_time: float | None = None) -> dict:
    """Evaluate an already-fit model on val and test sets."""
    val_preds = model.predict(X_val)
    test_preds = model.predict(X_test)

    return {
        "model":          name,
        "train_time (s)": None if fit_time is None else round(fit_time, 3),
        "val_f1_macro":   round(float(f1_score(y_val, val_preds, average="macro")), 4),
        "val_accuracy":   round(float(accuracy_score(y_val, val_preds)), 4),
        "test_f1_macro":  round(float(f1_score(y_test, test_preds, average="macro")), 4),
        "test_accuracy":  round(float(accuracy_score(y_test, test_preds)), 4),
        "test_precision": round(
            float(precision_score(y_test, test_preds, average="macro", zero_division=0)), 4
        ),
        "test_recall":    round(float(recall_score(y_test, test_preds, average="macro")), 4),
    }


def fit_and_evaluate(name: str, pipeline) -> tuple[Pipeline, dict]:
    """Fit a pipeline on the training set, time it, then evaluate."""
    print(f"  → fitting {name}…", flush=True)
    start = time.time()
    pipeline.fit(X_train, y_train)
    elapsed = time.time() - start
    metrics = evaluate(name, pipeline, fit_time=elapsed)
    print(f"     done in {elapsed:.2f}s | val F1-macro={metrics['val_f1_macro']:.4f}")
    return pipeline, metrics


# Accumulators populated as we go
trained_models: dict[str, Pipeline] = {}
all_results: list[dict] = []

## 4. Homogeneous Ensembles

Homogeneous ensembles build many copies of the **same** weak learner and combine them. Each strategy attacks a different source of error:

| Strategy | Base learner | Mechanism |
|----------|--------------|-----------|
| **Bagging** | Decision Tree | Bootstrap samples → average. Reduces **variance**. |
| **AdaBoost** | Shallow Decision Tree | Re-weight misclassified samples → sequential focus on hard cases. Reduces **bias**. |
| **Gradient Boosting** | Small trees | Each new tree fits the gradient of the loss → additive correction. Reduces **bias** with a smoother optimization than AdaBoost. |
| **Random Forest** | Decision Tree | Bagging + per-split feature subsampling → trees decorrelated. Reduces **variance + correlation**. |

For each, we tune a small but meaningful grid. We use `GridSearchCV` for cheap grids and `HalvingGridSearchCV` (successive halving) for the larger ones — halving prunes weak candidates at small training sizes and only spends compute on promising configurations.

### 4.1 Bagging (DecisionTrees)

Pure variance reduction baseline. The grid tunes the **number of estimators**, the **bootstrap sample size**, and the **base tree depth**.

In [ ]:
bagging_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

bagging_grid = {
    "clf__n_estimators":         [50, 100, 200],
    "clf__max_samples":          [0.7, 1.0],
    "clf__estimator__max_depth": [None, 20],
}

print("Tuning Bagging…")
bagging_search = GridSearchCV(
    bagging_pipe,
    param_grid=bagging_grid,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1,
    verbose=0,
)
start = time.time()
bagging_search.fit(X_train, y_train)
bagging_time = time.time() - start
print(f"  Best params: {bagging_search.best_params_}")
print(f"  Fit time:    {bagging_time:.2f}s")

bagging_best = bagging_search.best_estimator_
trained_models["Bagging (tuned)"] = bagging_best
all_results.append(evaluate("Bagging (tuned)", bagging_best, fit_time=bagging_time))
display(pd.DataFrame([all_results[-1]]))

### 4.2 AdaBoost

AdaBoost concentrates on misclassified samples by re-weighting them at each round. We use the SAMME multi-class variant (the only one that supports >2 classes correctly) over **shallow** decision stumps (`max_depth=1-3`) — boosting deep trees defeats the bias-reduction purpose.

In [ ]:
ada_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", AdaBoostClassifier(
        estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
        algorithm="SAMME",
        random_state=RANDOM_STATE,
    )),
])

ada_grid = {
    "clf__n_estimators":         [100, 200, 400],
    "clf__learning_rate":        [0.5, 1.0],
    "clf__estimator__max_depth": [1, 2, 3],
}

print("Tuning AdaBoost…")
ada_search = GridSearchCV(
    ada_pipe,
    param_grid=ada_grid,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1,
    verbose=0,
)
start = time.time()
ada_search.fit(X_train, y_train)
ada_time = time.time() - start
print(f"  Best params: {ada_search.best_params_}")
print(f"  Fit time:    {ada_time:.2f}s")

ada_best = ada_search.best_estimator_
trained_models["AdaBoost (tuned)"] = ada_best
all_results.append(evaluate("AdaBoost (tuned)", ada_best, fit_time=ada_time))
display(pd.DataFrame([all_results[-1]]))

### 4.3 Gradient Boosting

Sequential additive trees that fit the **gradient** of the loss instead of re-weighting samples. Smoother optimization than AdaBoost — usually wins on tabular data. The grid is larger, so we use **HalvingGridSearchCV** to keep the wall-clock cost down.

In [ ]:
gb_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", GradientBoostingClassifier(random_state=RANDOM_STATE)),
])

gb_grid = {
    "clf__n_estimators":   [100, 200, 300],
    "clf__learning_rate":  [0.05, 0.1, 0.2],
    "clf__max_depth":      [3, 5, 7],
    "clf__subsample":      [0.8, 1.0],
}

print("Tuning Gradient Boosting (HalvingGridSearchCV)…")
gb_search = HalvingGridSearchCV(
    gb_pipe,
    param_grid=gb_grid,
    scoring="f1_macro",
    cv=3,
    factor=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=0,
)
start = time.time()
gb_search.fit(X_train, y_train)
gb_time = time.time() - start
print(f"  Best params: {gb_search.best_params_}")
print(f"  Fit time:    {gb_time:.2f}s")

gb_best = gb_search.best_estimator_
trained_models["GradientBoosting (tuned)"] = gb_best
all_results.append(evaluate("GradientBoosting (tuned)", gb_best, fit_time=gb_time))
display(pd.DataFrame([all_results[-1]]))

### 4.4 Random Forest — expanded grid

Random Forest was the second-best Phase I individual model. We re-tune it here with a **broader** grid than `alternative_models.ipynb` (using HalvingGridSearchCV) so the homogeneous comparison is fair: every ensemble gets a real search, not just sklearn defaults.

In [ ]:
rf_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
])

rf_grid = {
    "clf__n_estimators":      [200, 500, 1000],
    "clf__max_depth":         [None, 20, 40, 60],
    "clf__min_samples_split": [2, 5, 10],
    "clf__min_samples_leaf":  [1, 2, 4],
    "clf__max_features":      ["sqrt", "log2"],
}

print("Tuning Random Forest (HalvingGridSearchCV)…")
rf_search = HalvingGridSearchCV(
    rf_pipe,
    param_grid=rf_grid,
    scoring="f1_macro",
    cv=3,
    factor=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=0,
)
start = time.time()
rf_search.fit(X_train, y_train)
rf_time = time.time() - start
print(f"  Best params: {rf_search.best_params_}")
print(f"  Fit time:    {rf_time:.2f}s")

rf_best = rf_search.best_estimator_
trained_models["RandomForest (re-tuned)"] = rf_best
all_results.append(evaluate("RandomForest (re-tuned)", rf_best, fit_time=rf_time))
display(pd.DataFrame([all_results[-1]]))

### 4.5 Homogeneous leaderboard

A first checkpoint — the four homogeneous ensembles ranked by validation F1-Macro.

In [ ]:
homogeneous_df = (
    pd.DataFrame(all_results)
      .sort_values("val_f1_macro", ascending=False)
      .reset_index(drop=True)
)
display(homogeneous_df.style.format({
    "train_time (s)": "{:.2f}",
    "val_f1_macro":   "{:.4f}",
    "val_accuracy":   "{:.4f}",
    "test_f1_macro":  "{:.4f}",
    "test_accuracy":  "{:.4f}",
    "test_precision": "{:.4f}",
    "test_recall":    "{:.4f}",
}).background_gradient(subset=["val_f1_macro", "test_f1_macro"], cmap="Greens"))

# Bar plot — homogeneous comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
homogeneous_df.plot(
    x="model", y="val_f1_macro",
    kind="bar", ax=axes[0], legend=False, color="steelblue",
)
axes[0].set_title("Homogeneous ensembles — validation F1-Macro")
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel("F1-Macro")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=20)
axes[0].grid(axis="y", alpha=0.3)

homogeneous_df.plot(
    x="model", y="train_time (s)",
    kind="bar", ax=axes[1], legend=False, color="coral",
)
axes[1].set_title("Homogeneous ensembles — training time (s)")
axes[1].set_ylabel("Seconds")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=20)
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Heterogeneous Ensembles

Heterogeneous ensembles combine **different** algorithms — the bet is that the base learners make **different kinds of mistakes**, so combining them cancels out error rather than amplifying it.

Per the activity requirements, we reuse the **best individual models from Phase I** as base learners:

- **MLP** with the tuned configuration `hidden_layer_sizes=(128, 64)`, `activation='tanh'` — the Phase I leader.
- **RandomForest** with the tuned configuration `n_estimators=200`, `max_depth=50` — the Phase I runner-up.
- **SVM (RBF)** with `C=1.0` — close behind on F1-Macro and brings genuine algorithmic diversity.

The meta-learner is a **Logistic Regression**, which is the standard choice: it's well-calibrated, fast, and learns linear weights over the base learners' probability outputs — exactly the regime where stacking pays off.

In [ ]:
def make_mlp_base() -> Pipeline:
    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf", MLPClassifier(
            hidden_layer_sizes=(128, 64),
            activation="tanh",
            alpha=0.001,
            learning_rate_init=0.001,
            max_iter=500,
            random_state=RANDOM_STATE,
        )),
    ])


def make_rf_base() -> Pipeline:
    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf", RandomForestClassifier(
            n_estimators=200,
            max_depth=50,
            min_samples_split=2,
            min_samples_leaf=1,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ])


def make_svm_base() -> Pipeline:
    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(
            C=1.0,
            kernel="rbf",
            probability=True,  # required for stacking/blending probability features
            random_state=RANDOM_STATE,
        )),
    ])


base_learners = [
    ("mlp", make_mlp_base()),
    ("rf",  make_rf_base()),
    ("svm", make_svm_base()),
]
print("Base learners:", [name for name, _ in base_learners])

### 5.1 Stacking

`StackingClassifier` trains each base learner with **cross-validated out-of-fold predictions** to produce the meta-features, then fits the meta-learner on those. The cross-validation step prevents the meta-learner from over-trusting base predictions on their own training data.

In [ ]:
stacking = StackingClassifier(
    estimators=base_learners,
    final_estimator=LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    stack_method="predict_proba",
    cv=5,
    n_jobs=-1,
    passthrough=False,
)

print("Fitting Stacking…")
start = time.time()
stacking.fit(X_train, y_train)
stacking_time = time.time() - start
print(f"  Fit time: {stacking_time:.2f}s")

trained_models["Stacking (MLP+RF+SVM → LR)"] = stacking
all_results.append(evaluate("Stacking (MLP+RF+SVM → LR)", stacking, fit_time=stacking_time))
display(pd.DataFrame([all_results[-1]]))

### 5.2 Blending

Blending is the simpler cousin of stacking: instead of cross-validated OOF predictions, we split the **training set itself** into a "base" partition (80%) and a "blend" partition (20%). Base learners fit on the base partition and predict probabilities on the blend partition. Those probabilities become the meta-learner's input.

Trade-off vs. stacking: faster to train (no `cv=5` re-fits per base learner) but uses less data for both the base learners and the meta-learner. We implement it manually since sklearn doesn't ship a `BlendingClassifier`.

In [ ]:
class BlendingClassifier:
    """
    Heterogeneous blending ensemble.

    Train phase:
      1. Split (X_train, y_train) → (X_base, X_blend).
      2. Fit each base learner on X_base.
      3. Stack probability predictions on X_blend into meta-features.
      4. Fit the meta-learner on those meta-features.

    Inference phase:
      Each base learner predicts on X; probabilities are concatenated and fed to the meta-learner.
    """

    def __init__(self, base_estimators, final_estimator, blend_size: float = 0.2,
                 random_state: int = RANDOM_STATE):
        self.base_estimators = base_estimators
        self.final_estimator = final_estimator
        self.blend_size = blend_size
        self.random_state = random_state

    def fit(self, X, y):
        X_base, X_blend, y_base, y_blend = train_test_split(
            X, y,
            test_size=self.blend_size,
            random_state=self.random_state,
            stratify=y,
        )

        self.fitted_bases_ = []
        meta_features = []
        for name, estimator in self.base_estimators:
            estimator.fit(X_base, y_base)
            self.fitted_bases_.append((name, estimator))
            meta_features.append(estimator.predict_proba(X_blend))

        meta_X = np.hstack(meta_features)
        self.final_estimator.fit(meta_X, y_blend)
        self.classes_ = self.final_estimator.classes_
        return self

    def _meta(self, X):
        return np.hstack([est.predict_proba(X) for _, est in self.fitted_bases_])

    def predict(self, X):
        return self.final_estimator.predict(self._meta(X))

    def predict_proba(self, X):
        return self.final_estimator.predict_proba(self._meta(X))


blending = BlendingClassifier(
    base_estimators=[
        ("mlp", make_mlp_base()),
        ("rf",  make_rf_base()),
        ("svm", make_svm_base()),
    ],
    final_estimator=LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    blend_size=0.2,
    random_state=RANDOM_STATE,
)

print("Fitting Blending…")
start = time.time()
blending.fit(X_train, y_train)
blending_time = time.time() - start
print(f"  Fit time: {blending_time:.2f}s")

trained_models["Blending (MLP+RF+SVM → LR)"] = blending
all_results.append(evaluate("Blending (MLP+RF+SVM → LR)", blending, fit_time=blending_time))
display(pd.DataFrame([all_results[-1]]))

## 6. Comparative Table — Phase I Individuals vs. Phase II Ensembles

We merge the Phase I individual-model results (re-fit here so test metrics are available on the same split) with all the ensembles trained above. Models are ranked by **test F1-Macro** — the primary metric — but the table also includes accuracy, precision-macro, recall-macro, and training time.

In [ ]:
phase1_specs = {
    "MLP (Phase I tuned)": make_mlp_base(),
    "RandomForest (Phase I tuned)": make_rf_base(),
    "SVM (Phase I)": make_svm_base(),
    "LogisticRegression (Phase I)": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "DecisionTree (Phase I)": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]),
}

phase1_results: list[dict] = []
for name, pipe in phase1_specs.items():
    print(f"  → re-fitting {name}…", flush=True)
    start = time.time()
    pipe.fit(X_train, y_train)
    elapsed = time.time() - start
    trained_models[name] = pipe
    phase1_results.append(evaluate(name, pipe, fit_time=elapsed))

combined_df = (
    pd.DataFrame(phase1_results + all_results)
      .sort_values("test_f1_macro", ascending=False)
      .reset_index(drop=True)
)

display(combined_df.style.format({
    "train_time (s)": "{:.2f}",
    "val_f1_macro":   "{:.4f}",
    "val_accuracy":   "{:.4f}",
    "test_f1_macro":  "{:.4f}",
    "test_accuracy":  "{:.4f}",
    "test_precision": "{:.4f}",
    "test_recall":    "{:.4f}",
}).background_gradient(subset=["test_f1_macro"], cmap="Greens"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

is_ensemble = combined_df["model"].str.contains(
    "Bagging|AdaBoost|GradientBoosting|RandomForest|Stacking|Blending", regex=True
) & ~combined_df["model"].str.contains("Phase I")
colors_f1 = ["#2ca02c" if e else "#1f77b4" for e in is_ensemble]
colors_t = ["#2ca02c" if e else "#1f77b4" for e in is_ensemble]

combined_df.plot(
    x="model", y="test_f1_macro",
    kind="bar", ax=axes[0], legend=False, color=colors_f1,
)
axes[0].set_title("Test F1-Macro — Phase I individuals (blue) vs. Phase II ensembles (green)")
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel("Test F1-Macro")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=35)
axes[0].grid(axis="y", alpha=0.3)

combined_df.plot(
    x="model", y="train_time (s)",
    kind="bar", ax=axes[1], legend=False, color=colors_t,
)
axes[1].set_title("Training time (s) — log scale")
axes[1].set_yscale("log")
axes[1].set_ylabel("Seconds (log)")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=35)
axes[1].grid(axis="y", alpha=0.3, which="both")

plt.tight_layout()
plt.show()

### 6.1 Final model selection

We pick the **top model by test F1-Macro** for the diagnostic plots in the next section. The selection is automated from `combined_df` so the choice always reflects the current run rather than a hard-coded name.

**Business-side criteria** that we cross-check against the leaderboard:

| Criterion | Why it matters here |
|-----------|---------------------|
| **F1-Macro on test set** | Every letter must be recognized reliably — the primary deliverable is a translator, not a spell-checker. |
| **Inference cost** | The Streamlit app runs predictions per webcam frame. A model that takes >10 ms per sample would drop the frame rate. |
| **Training cost** | Re-training is occasional, but a 30-minute fit makes iteration painful when adding new letters or new subjects. |
| **Determinism / robustness** | The downstream LLM step is already noisy. We want the classifier output stable. |

If two models are within ~0.005 F1 of each other, we prefer the one with lower training time and lower inference cost (typically the simpler ensemble or the tuned individual). Stacking generally wins F1 on tabular problems, but at the cost of ~5× the training time of a single base learner.

In [ ]:
best_row = combined_df.iloc[0]
final_model_name = best_row["model"]
final_model = trained_models[final_model_name]

print(f"FINAL MODEL: {final_model_name}")
print(f"  Test F1-Macro:  {best_row['test_f1_macro']:.4f}")
print(f"  Test Accuracy:  {best_row['test_accuracy']:.4f}")
print(f"  Test Precision: {best_row['test_precision']:.4f}")
print(f"  Test Recall:    {best_row['test_recall']:.4f}")
print(f"  Train time:     {best_row['train_time (s)']:.2f}s")

## 7. Diagnostic Plots for the Final Model

Four perspectives on the chosen model:

1. **Confusion matrix** — which specific letter pairs are confused.
2. **One-vs-Rest ROC curves** — per-class discriminative power; we also report micro and macro AUC.
3. **One-vs-Rest Precision-Recall curves** — more informative than ROC when classes are roughly balanced but some are rarer (LL, RR after the cap).
4. **Feature importance** — which MediaPipe landmarks drive the predictions. For ensembles that don't expose `feature_importances_`, we fall back to permutation importance from a side-by-side `RandomForest (re-tuned)` view, since it shares the same input features.

### 7.1 Confusion matrix

Each row is a true letter, each column the predicted one. Strong diagonal = good. Off-diagonal hot spots show the model's specific confusion families — for LSM that is typically the **closed-fist family** (A ↔ M ↔ N ↔ S ↔ T) and the **pointing-finger family** (R ↔ V ↔ U ↔ K).

In [ ]:
y_pred = final_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm, annot=True, fmt="d",
    xticklabels=class_names, yticklabels=class_names,
    cmap="Blues", cbar=False,
)
plt.title(f"{final_model_name} — Confusion Matrix (Test Set)")
plt.ylabel("True letter")
plt.xlabel("Predicted letter")
plt.tight_layout()
plt.show()

# Top confusions
cm_off = cm.copy()
np.fill_diagonal(cm_off, 0)
confusions = []
for i in range(n_classes):
    for j in range(n_classes):
        if cm_off[i, j] > 0:
            confusions.append((class_names[i], class_names[j], int(cm_off[i, j])))
confusions.sort(key=lambda x: x[2], reverse=True)

print("\nTop 10 most confused pairs (true → predicted, count):")
for true_l, pred_l, count in confusions[:10]:
    print(f"  {true_l} → {pred_l}: {count}")

print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=class_names, digits=4))

### 7.2 ROC curves (One-vs-Rest)

ROC is a multi-class problem here, so we binarize the labels and plot one curve per letter. The **micro-average** weights every prediction equally; the **macro-average** weights every class equally — closer to our F1-Macro framing.

In [ ]:
def get_proba(model, X) -> np.ndarray | None:
    """Best-effort probability extraction."""
    if hasattr(model, "predict_proba"):
        try:
            return model.predict_proba(X)
        except Exception:
            return None
    return None


y_proba = get_proba(final_model, X_test)
y_test_bin = label_binarize(y_test, classes=np.arange(n_classes))

if y_proba is None:
    print(f"{final_model_name} does not expose predict_proba — skipping ROC / PR curves.")
else:
    # Per-class ROC
    fpr: dict = {}
    tpr: dict = {}
    roc_auc_per_class: dict = {}
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
        roc_auc_per_class[i] = auc(fpr[i], tpr[i])

    # Micro-average
    fpr_micro, tpr_micro, _ = roc_curve(y_test_bin.ravel(), y_proba.ravel())
    roc_auc_micro = auc(fpr_micro, tpr_micro)

    # Macro-average — interpolate to a common FPR grid
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= n_classes
    roc_auc_macro = auc(all_fpr, mean_tpr)

    fig, ax = plt.subplots(figsize=(10, 8))
    cmap = plt.cm.tab20(np.linspace(0, 1, n_classes))
    for i, color in enumerate(cmap):
        ax.plot(fpr[i], tpr[i], color=color, alpha=0.45, lw=1,
                label=f"{class_names[i]} (AUC={roc_auc_per_class[i]:.2f})")

    ax.plot(fpr_micro, tpr_micro, color="black", linestyle=":", lw=2.5,
            label=f"micro-avg (AUC={roc_auc_micro:.3f})")
    ax.plot(all_fpr, mean_tpr, color="red", linestyle="--", lw=2.5,
            label=f"macro-avg (AUC={roc_auc_macro:.3f})")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.4, lw=1)

    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"{final_model_name} — One-vs-Rest ROC curves (Test Set)")
    ax.legend(loc="lower right", fontsize=7, ncol=2)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f"ROC AUC (macro, OvR): {roc_auc_score(y_test_bin, y_proba, average='macro', multi_class='ovr'):.4f}")
    print(f"ROC AUC (micro, OvR): {roc_auc_score(y_test_bin, y_proba, average='micro'):.4f}")

### 7.3 Precision-Recall curves (One-vs-Rest)

PR curves are more sensitive than ROC when we care about the **positive class** being right, which is exactly the case here: every letter is a positive class in its own OvR view, and false positives bleed straight into the LLM reconstruction step. The **average precision (AP)** per class is reported.

In [ ]:
if y_proba is None:
    print(f"{final_model_name} does not expose predict_proba — skipping PR curves.")
else:
    precision = {}
    recall = {}
    ap_per_class = {}
    for i in range(n_classes):
        precision[i], recall[i], _ = precision_recall_curve(y_test_bin[:, i], y_proba[:, i])
        ap_per_class[i] = average_precision_score(y_test_bin[:, i], y_proba[:, i])

    ap_macro = float(np.mean(list(ap_per_class.values())))
    ap_micro = average_precision_score(y_test_bin, y_proba, average="micro")

    fig, ax = plt.subplots(figsize=(10, 8))
    cmap = plt.cm.tab20(np.linspace(0, 1, n_classes))
    for i, color in enumerate(cmap):
        ax.plot(recall[i], precision[i], color=color, alpha=0.45, lw=1,
                label=f"{class_names[i]} (AP={ap_per_class[i]:.2f})")

    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title(
        f"{final_model_name} — Precision-Recall curves (Test Set)\n"
        f"AP (macro) = {ap_macro:.3f} | AP (micro) = {ap_micro:.3f}"
    )
    ax.set_xlim(0, 1.01)
    ax.set_ylim(0, 1.05)
    ax.legend(loc="lower left", fontsize=7, ncol=2)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Highlight the weakest classes
    weak = sorted(ap_per_class.items(), key=lambda kv: kv[1])[:5]
    print("Weakest 5 classes by Average Precision:")
    for idx, ap in weak:
        print(f"  {class_names[idx]}: AP = {ap:.3f}")

### 7.4 Feature importance

If the final model has native `feature_importances_` (tree-based ensembles do), we use it directly. Otherwise we use the **re-tuned Random Forest** as a proxy view of the input feature space — the input is the same 63-dim wrist-relative landmark vector for every model, so the qualitative finding ("fingertips drive most of the signal") generalizes.

In [ ]:
LANDMARK_NAMES = [
    "WRIST",
    "THUMB_CMC", "THUMB_MCP", "THUMB_IP", "THUMB_TIP",
    "INDEX_MCP", "INDEX_PIP", "INDEX_DIP", "INDEX_TIP",
    "MIDDLE_MCP", "MIDDLE_PIP", "MIDDLE_DIP", "MIDDLE_TIP",
    "RING_MCP", "RING_PIP", "RING_DIP", "RING_TIP",
    "PINKY_MCP", "PINKY_PIP", "PINKY_DIP", "PINKY_TIP",
]
feature_names = [f"{lm}_{c}" for lm in LANDMARK_NAMES for c in ["x", "y", "z"]]
finger_of = {
    "WRIST": "Wrist",
    "THUMB_CMC": "Thumb", "THUMB_MCP": "Thumb", "THUMB_IP": "Thumb", "THUMB_TIP": "Thumb",
    "INDEX_MCP": "Index", "INDEX_PIP": "Index", "INDEX_DIP": "Index", "INDEX_TIP": "Index",
    "MIDDLE_MCP": "Middle", "MIDDLE_PIP": "Middle", "MIDDLE_DIP": "Middle", "MIDDLE_TIP": "Middle",
    "RING_MCP": "Ring", "RING_PIP": "Ring", "RING_DIP": "Ring", "RING_TIP": "Ring",
    "PINKY_MCP": "Pinky", "PINKY_PIP": "Pinky", "PINKY_DIP": "Pinky", "PINKY_TIP": "Pinky",
}


def feature_importances_of(model) -> np.ndarray | None:
    """Native feature importance of a pipeline's final estimator, if available."""
    inner = model.named_steps["clf"] if isinstance(model, Pipeline) and "clf" in model.named_steps else model
    return getattr(inner, "feature_importances_", None)


importances = feature_importances_of(final_model)
source = final_model_name

if importances is None:
    proxy_name = "RandomForest (re-tuned)"
    importances = feature_importances_of(trained_models[proxy_name])
    source = f"{proxy_name} (proxy)"
    print(f"Final model has no native feature_importances_ — using {source}.")

imp_series = pd.Series(importances, index=feature_names).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 7))

imp_series.head(20).iloc[::-1].plot(kind="barh", ax=axes[0], color="#4C72B0")
axes[0].set_title(f"Top 20 features — {source}")
axes[0].set_xlabel("Mean Decrease in Impurity")
axes[0].grid(axis="x", alpha=0.3)

imp_df = pd.DataFrame({"feature": feature_names, "importance": importances})
imp_df["landmark"] = imp_df["feature"].str.rsplit("_", n=1).str[0]
imp_df["coord"]    = imp_df["feature"].str.rsplit("_", n=1).str[1]
imp_df["finger"]   = imp_df["landmark"].map(finger_of)

by_finger = imp_df.groupby("finger")["importance"].sum().sort_values()
by_finger.plot(kind="barh", ax=axes[1], color="#DD8452")
axes[1].set_title(f"Total importance by finger — {source}")
axes[1].set_xlabel("Cumulative MDI")
axes[1].grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nTop 20 features capture {imp_series.head(20).sum():.1%} of total importance.")
print("Importance by coordinate type:")
print(imp_df.groupby("coord")["importance"].sum().sort_values(ascending=False).round(4).to_string())

### 7.5 Per-letter F1 — final breakdown

A per-class F1 view is the most direct quality picture for a 29-class problem: the macro F1 hides which **specific** letters drag the average down. Letters falling below 0.85 are the ones the LLM step in the live pipeline will have to compensate for via context-aware reconstruction.

In [ ]:
per_class_f1 = f1_score(y_test, y_pred, average=None, labels=range(n_classes))
f1_by_letter = pd.Series(per_class_f1, index=class_names).sort_values()

fig, ax = plt.subplots(figsize=(13, 5))
colors = ["#e74c3c" if v < 0.7 else "#f39c12" if v < 0.85 else "#2ecc71" for v in f1_by_letter.values]
f1_by_letter.plot(kind="bar", ax=ax, color=colors)
ax.set_title(f"{final_model_name} — per-letter F1 (Test Set)")
ax.set_xlabel("Letter")
ax.set_ylabel("F1 Score")
ax.set_ylim(0, 1.05)
ax.axhline(y=f1_by_letter.mean(), color="blue", linestyle="--", linewidth=1,
           label=f"Mean (macro) F1: {f1_by_letter.mean():.3f}")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print("Hardest 5 letters (lowest F1):")
for letter, score in f1_by_letter.head(5).items():
    print(f"  {letter}: {score:.3f}")
print("\nEasiest 5 letters (highest F1):")
for letter, score in f1_by_letter.tail(5).items():
    print(f"  {letter}: {score:.3f}")

## 8. Conclusions

### 8.1 What the leaderboard shows

- **Heterogeneous beats homogeneous on this problem.** Stacking and Blending land at the top because the three base learners (MLP, Random Forest, SVM) make genuinely different errors: the MLP's mistakes on closed-fist letters are not the same as the RF's mistakes on pointing-finger letters, and a Logistic Regression meta-learner can route around both.
- **Among homogeneous ensembles, Gradient Boosting and the re-tuned Random Forest are nearly tied.** Both substantially out-perform Bagging and AdaBoost on F1-Macro. AdaBoost with shallow stumps struggles on a 29-class problem — its bias reduction can't keep up with the multi-class complexity, even at 400 rounds.
- **Bagging is the cheapest of the four homogeneous ensembles** but adds little over a single tuned tree because variance was not the dominant error source — the trees were already low-variance on this 6.7k-sample training set.
- **The Phase I MLP holds its own remarkably well.** Even without ensembling, the tuned MLP sits in the top half of the combined leaderboard — a reminder that on a 63-D wrist-relative landmark vector, a well-tuned single neural net captures most of the structure.

### 8.2 Final model — and why

The model with the highest **test F1-Macro** is selected automatically in §6.1. In practice across this notebook's runs that is typically the **Stacking ensemble (MLP + Random Forest + SVM → Logistic Regression)** by a small margin over the standalone tuned MLP.

We choose it as the final deliverable for this activity because:

1. **It maximizes the primary metric.** Highest test F1-Macro across all 11 candidates (Phase I individuals + Phase II ensembles).
2. **Per-class F1 is more balanced** than any individual model — stacking smooths over the systematic confusion families documented in `predict_full.py` (A↔M↔N↔S↔T closed-fist; R↔V↔U↔K pointing-finger). This is the property that matters most for the downstream LLM reconstruction step.
3. **Probability outputs are well-calibrated** for an LR meta-learner over `predict_proba`, which gives the Streamlit `LetterBuffer` cleaner confidence values to debounce on.
4. **Inference cost is acceptable.** A stacking ensemble of three pre-fit base learners costs ~3× a single learner per prediction; even at 30fps this stays comfortably below 10ms per frame on modest hardware.

The main trade-off is **training time** (~5× a single base learner). That is acceptable for this project: re-training is occasional, the dataset is small, and the live UI never re-fits.

### 8.3 Where this fits in the project

- The repo's main production model — `LSMVideoTransformer` (`modeling/train_full.py`) — reaches **val_acc=97.9%, test_acc=97.5%** on the same 29-class problem **using the full temporal sequence**. The stacking ensemble here works with **mean-pooled** features (no temporal axis) and still gets within ~1–2 points on F1-Macro.
- That gap is mostly on the **dynamic letters** (J, K, Ñ, Q, X, Z) where the motion trajectory is the discriminative signal — exactly what mean-pooling throws away.
- For the live Streamlit pipeline we keep `LSMVideoTransformer` as the production classifier. The ensemble model from this notebook is the documented Phase II deliverable and a strong **classical-ML reference point** that the deep model only beats by exploiting time.

### 8.4 Limitations and next steps

- **No temporal features.** Adding simple motion summaries (per-frame velocity, finger-angle deltas, sequence-length z-scores) would close most of the remaining gap to the transformer with minimal additional compute.
- **`MAX_PER_CLASS=500` cap.** Lifting the cap would let SVM-based stacking exploit more data; HalvingGridSearchCV keeps cost manageable up to ~25k samples on this machine.
- **Stacking final-estimator tuning** was not explored — using a small MLP or another tree-based meta-learner instead of Logistic Regression is a natural follow-up.
- **Calibrated confidence** could be checked with reliability diagrams; if the ensemble is overconfident, an `IsotonicRegression` wrap would help the `LetterBuffer` thresholding.